In [20]:
!pip install pandas==1.3.5 tensorflow==2.12.0 numpy==1.23.5 matplotlib==3.5.3 scikit-learn==1.4.2
#numpy==1.22.4
#numpy==1.21.6

  Using cached tensorboard_data_server-0.7.2-py3-none-manylinux_2_31_x86_64.whl.metadata (1.1 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 585.9/585.9 MB 11.0 MB/s eta 0:00:0000:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 14.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 17.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 MB 14.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 17.4 MB/s eta 0:00:00a 0:00:01
Using cached tensorboard_data_server-0.7.2-py3-none-manylinux_2_31_x86_64.whl (6.6 MB)
  Attempting uninstall: flatbuffers
    Found existing installation: flatbuffers 1.12
    Uninstalling flatbuffers-1.12:
      Successfully uninstalled flatbuffers-1.12
  Attempting uninstall: wrapt
    Found existing in

In [32]:
import pandas as pd 
#2.9.0 tensorflow
import numpy as np


In [33]:
# Cargar el archivo CSV en un DataFrame
df = pd.read_csv("temperaturas.csv")
df = df.drop_duplicates()
# Mostrar las primeras filas del DataFrame
print(df.head())

                    city                      date  tmed  prec
0                 MURCIA  1990-01-01T00:00:00.000Z  12.0     0
1              CARTAGENA  1990-01-01T00:00:00.000Z  13.0     0
2  SAN JAVIER AEROPUERTO  1990-01-01T00:00:00.000Z  12.0     0
3                  LORCA  1990-01-01T00:00:00.000Z  12.0     0
4                 MURCIA  1990-01-02T00:00:00.000Z  12.0     2


In [34]:
registros_por_ciudad = df.groupby("city").size()
print(registros_por_ciudad)

city
CARTAGENA                12736
LORCA                    12626
MURCIA                   12826
SAN JAVIER AEROPUERTO    12826
dtype: int64


In [35]:
import pandas as pd
import numpy as np

# 2. Convertir la columna "date" a tipo datetime
df['date'] = pd.to_datetime(df['date'])

# 3. Pivotear el DataFrame: filas = ciudades, columnas = fechas, valores = tmed
df_pivot = df.pivot(index='city', columns='date', values='tmed')

# 4. Crear un rango completo de fechas (diario)
fecha_inicio = df['date'].min()
fecha_fin = df['date'].max()
all_dates = pd.date_range(start=fecha_inicio, end=fecha_fin, freq='D')

# Reindexar las columnas del DataFrame para incluir todas las fechas
df_pivot = df_pivot.reindex(columns=all_dates)

# 5. Función para rellenar valores faltantes con la media de los 5 días anteriores
def fill_missing_with_rolling_mean(row, window=5):
    """
    Para cada valor faltante (NaN) en la serie (row), se calcula la media de los
    window días anteriores (si existen datos) y se asigna ese valor.
    """
    # Recorrer la serie en orden cronológico
    for idx in range(len(row)):
        if pd.isna(row.iloc[idx]):
            # Si es el primer día o no hay días anteriores, se salta
            if idx == 0:
                continue
            # Determinar el índice de inicio para el window (máximo 5 días previos)
            start_idx = max(0, idx - window)
            # Seleccionar los valores de los días anteriores
            prev_vals = row.iloc[start_idx:idx]
            # Si hay al menos un valor no nulo, calcular la media
            if not prev_vals.dropna().empty:
                mean_val = prev_vals.dropna().mean()
                #row.iloc[idx] = mean_val
                row.iloc[idx] = int(round(mean_val))

    return row

# Aplicar la función a cada fila (cada ciudad)
df_pivot = df_pivot.apply(fill_missing_with_rolling_mean, axis=1)

# 6. Visualizar el DataFrame resultante
print(df_pivot.head())

df_pivot.to_csv("temperaturas_spain_pivot.csv", index= True)


                       1990-01-01 00:00:00+00:00  1990-01-02 00:00:00+00:00  \
city                                                                          
CARTAGENA                                   13.0                       15.0   
LORCA                                       12.0                       11.0   
MURCIA                                      12.0                       12.0   
SAN JAVIER AEROPUERTO                       12.0                       14.0   

                       1990-01-03 00:00:00+00:00  1990-01-04 00:00:00+00:00  \
city                                                                          
CARTAGENA                                   12.0                       12.0   
LORCA                                        9.0                       10.0   
MURCIA                                       9.0                       11.0   
SAN JAVIER AEROPUERTO                       10.0                       12.0   

                       1990-01-05 00:00:00+00:00  

In [36]:
# df_pivot.describe()
for index, row in df_pivot.iterrows():
    print("La fila {} tiene {} columnas".format(index, len(row)))

La fila CARTAGENA tiene 12826 columnas
La fila LORCA tiene 12826 columnas
La fila MURCIA tiene 12826 columnas
La fila SAN JAVIER AEROPUERTO tiene 12826 columnas


In [90]:
with open("temperaturas_spain_pivot2.csv", "r") as datafile:
  # Leemos todas las líneas del archivo CSV y las guardamos en la lista ts_list_train.
  # Cada línea del archivo corresponderá a una cadena de caracteres en esta lista.
  ts_list_train = datafile.readlines()
  # Convertimos la lista de cadenas ts_list_train en un array de numpy.
  # rstrip elimina espacios en blanco  y saltos de linea
  # El contenido del array se crea utilizando una compresión de lista, donde cada cadena
  # de la lista original se divide en valores separados por comas (suponiendo que el
  # archivo CSV esté en formato estándar), se elimina cualquier carácter de nueva línea ('\n')
  # y se convierte en un array de numpy con tipo de dato float32.
  ts_list_train = np.asarray([np.asarray(l.rstrip().split(','),
                      dtype=np.float32) for l in ts_list_train])


In [91]:
len(ts_list_train)

4

Confirmamos que tenemos también **12 filas (una por ciudad)**. Creamos lista de ciudades y momento inicial de la serie temporal:

In [92]:
headers = [
  'CARTAGENA',
  'LORCA',
  'MURCIA',
  'SAN JAVIER AEROPUERTO'
]
import datetime
# Fecha inicial:  1 de octubre de 2012 a las 13:00 horas.1990-01-01
initial_moment = datetime.datetime(1990, 1, 1, 0, 0)

import matplotlib.pyplot as plt
# get_cmap() de plt sirve para obtener una paleta de colores.
# Esta función toma dos argumentos:
#   · El nombre de la paleta de colores ('Set3')
#   · El número de colores que se desean (longitud de headers)#
colors = plt.get_cmap('Set3', len(headers))

 A continuación vemos cuantas **columnas (valores tomados en cada ciudad)**:

In [93]:
for i in range(len(headers)):
  print("Longitud de la serie {0}: {1} dias. De {3} a {2}".
  format(headers[i], ts_list_train[i].shape[0],
          initial_moment +
          datetime.timedelta(days=ts_list_train[i].shape[0]-1),
          initial_moment))

Longitud de la serie CARTAGENA: 12826 dias. De 1990-01-01 00:00:00 a 2025-02-11 00:00:00
Longitud de la serie LORCA: 12826 dias. De 1990-01-01 00:00:00 a 2025-02-11 00:00:00
Longitud de la serie MURCIA: 12826 dias. De 1990-01-01 00:00:00 a 2025-02-11 00:00:00
Longitud de la serie SAN JAVIER AEROPUERTO: 12826 dias. De 1990-01-01 00:00:00 a 2025-02-11 00:00:00
